# TechnoStamp - análisis integral

Notebook orquestador: explica y ejecuta las etapas existentes sin duplicar la lógica de `04_scripts/`.


## Cómo usar este notebook

Ejecuta las celdas de arriba hacia abajo. Cada etapa corre como un proceso Python independiente usando el mismo entorno del kernel. Los scripts mantienen la lógica y el notebook conserva la secuencia y el contexto.


In [9]:
from pathlib import Path
import subprocess
import sys
from IPython.display import Image, display

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "03_notebooks" else NOTEBOOK_DIR
SCRIPTS_DIR = PROJECT_ROOT / "04_scripts"
VISUALIZACIONES_DIR = PROJECT_ROOT / "06_resultados" / "Discovery" / "visualizaciones"

if not SCRIPTS_DIR.exists():
    raise FileNotFoundError(f"No se encontro la carpeta de scripts: {SCRIPTS_DIR}")

def _mostrar_salida(texto: str, stream) -> None:
    if texto:
        print(texto, end="" if texto.endswith("\n") else "\n", file=stream)

def mostrar_graficos(nombres: list[str], width: int = 1100) -> None:
    for nombre in nombres:
        imagen = VISUALIZACIONES_DIR / nombre
        if not imagen.exists():
            raise FileNotFoundError(f"No se encontro el grafico: {imagen}")
        display(Image(filename=str(imagen), width=width))


def ejecutar_etapa(nombre_script: str) -> None:
    script = SCRIPTS_DIR / nombre_script

    if not script.exists():
        raise FileNotFoundError(f"No existe el script: {script}")

    ejecutor = "\n".join([
        "import pandas as pd, runpy, sys",
        "pd.set_option('display.float_format', lambda x: format(x, ',.10f').rstrip('0').rstrip('.'))",
        "runpy.run_path(sys.argv[1], run_name='__main__')",
    ])
    resultado = subprocess.run(
        [sys.executable, "-c", ejecutor, str(script)],
        cwd=PROJECT_ROOT,
        text=True,
        encoding="utf-8",
        errors="strict",
        capture_output=True,
        check=False,
    )

    _mostrar_salida(resultado.stdout, sys.stdout)
    _mostrar_salida(resultado.stderr, sys.stderr)

    if resultado.returncode != 0:
        raise subprocess.CalledProcessError(
            resultado.returncode,
            resultado.args,
            output=resultado.stdout,
            stderr=resultado.stderr,
        )

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Scripts: {SCRIPTS_DIR}")


Proyecto: C:\Users\Dell\Agus\Nivii AI
Scripts: C:\Users\Dell\Agus\Nivii AI\04_scripts


## 1. Perfilado inicial

Describe la estructura de las tres fuentes: columnas, tipos y valores faltantes. Sirve para entender el material antes de tomar decisiones.

**Script ejecutado:** `04_scripts/01_perfilado.py`


In [10]:
ejecutar_etapa("01_perfilado.py")


TABLA: empleados_mensual   shape=(9733, 47)
------------------------------------------------------------------------------------------
                                dtype  n_nulos  pct_nulos  n_unicos               ejemplo
empleado_id                     int64        0        0.0       694                  1000
mes_snapshot                      str        0        0.0        17            2024-01-01
activo                           bool        0        0.0         2                  True
nombre                            str        0        0.0       596         Teresa Carmen
genero                            str        0        0.0         2                     F
edad                            int64        0        0.0        41                    62
fecha_nacimiento                  str        0        0.0       127            1962-06-15
fecha_ingreso                     str        0        0.0       103            2012-01-01
antiguedad_meses                int64        0        0

## 2. Calidad de datos

Revisa encoding, granularidad, duplicados, coherencia entre fechas y valores fuera de rango. Detectar estos problemas antes de analizar evita conclusiones engañosas.

**Script ejecutado:** `04_scripts/02_calidad.py`


In [11]:
ejecutar_etapa("02_calidad.py")


### 1. ENCODING
  utf-8      OK -> puesto ejemplo: 'Técnico Setup'
  latin-1    OK -> puesto ejemplo: 'TÃ©cnico Setup'
  cp1252     FALLA: UnicodeDecodeError

### 2. GRANULARIDAD empleados_mensual
  filas: 9733 | empleados unicos: 694 | meses: 17
  duplicados (empleado_id, mes_snapshot): 0
  rango meses: 2024-01-01 -> 2025-05-01

  headcount ACTIVO por mes:
              filas  activos
mes_snapshot                
2024-01-01      605      586
2024-02-01      587      587
2024-03-01      591      574
2024-04-01      577      568
2024-05-01      572      567
2024-06-01      571      564
2024-07-01      570      566
2024-08-01      572      562
2024-09-01      566      560
2024-10-01      565      562
2024-11-01      567      563
2024-12-01      566      552
2025-01-01      558      554
2025-02-01      560      553
2025-03-01      563      559
2025-04-01      577      562
2025-05-01      566      562

### 3. CONSISTENCIA INTERNA
  edad declarada vs calculada -> dif abs >1 anio: 51
  antig

## 3. Limpieza y preparación - versión mejorada

Normaliza nombres y tipos, documenta por qué faltan datos y marca inconsistencias sin borrar información. 
**Script ejecutado:** `04_scripts/13_limpieza_v2.py`


In [12]:
ejecutar_etapa("13_limpieza_v2.py")


Carga OK — empleados (9733, 47) | eventos (8705, 18) | capacitaciones (729, 17)

MEJORA 3 — nombres de columna sin eñes ni acentos
  [renombrar_columna] {'tabla': 'empleados', 'de': 'años_en_puesto_actual', 'a': 'anios_en_puesto_actual'}
  [renombrar_columna] {'tabla': 'empleados', 'de': 'horas_training_acumulado_año', 'a': 'horas_training_acumulado_anio'}
  Nombres finales verificados como unicos en las 3 tablas

MEJORA 4 — tipos correctos
  [a_fecha] {'tabla': 'empleados', 'col': 'mes_snapshot'}
  [a_fecha] {'tabla': 'empleados', 'col': 'fecha_nacimiento'}
  [a_fecha] {'tabla': 'empleados', 'col': 'fecha_ingreso'}
  [a_fecha] {'tabla': 'empleados', 'col': 'fecha_salida'}
  [a_fecha] {'tabla': 'empleados', 'col': 'ultimo_training_fecha'}
  [a_fecha] {'tabla': 'eventos', 'col': 'fecha_evento'}
  [a_fecha] {'tabla': 'capacitaciones', 'col': 'fecha_inicio'}
  [a_fecha] {'tabla': 'capacitaciones', 'col': 'fecha_fin'}
  [a_entero_con_vacios] {'tabla': 'empleados', 'col': 'manager_id', 'dec

## 4. P1 y P2 - sucesión y equidad

Calcula el riesgo de sucesión por puesto y describe la distribución de género por área y nivel. Ayuda a encontrar posiciones críticas y posibles brechas.

**Script ejecutado:** `04_scripts/04_p1_p2.py`


In [13]:
ejecutar_etapa("04_p1_p2.py")


Universo de analisis: 562 empleados ACTIVOS a 2025-05-01

P1 — RIESGO DE SUCESION / JUBILACION

Jubilaciones proximas (sobre 562 activos):
  <= 12 meses   :   4 empleados ( 0.7%) | de ellos criticos: 4
  <= 24 meses   :   4 empleados ( 0.7%) | de ellos criticos: 4
  <= 36 meses   :   4 empleados ( 0.7%) | de ellos criticos: 4
  <= 60 meses   :   4 empleados ( 0.7%) | de ellos criticos: 4

Posiciones criticas activas: 5 (0.9% de la dotacion)

--- Criticos que se jubilan en <=12m, por area y puesto ---
                                   n  edad_prom  antig_anios  meses_rest    salario
area      puesto                                                                   
Logística Team Leader Logística    2       62.5          5.3           9  2316035.1
Estampado Técnico Setup            1       64.0         11.3           6  1716338.4
Logística Supervisor de Logística  1       64.0          4.3           6  3486692.6

  TOTAL criticos en riesgo <=12m: 4

--- Cobertura: para cada puesto crit

C:\Users\Dell\Agus\Nivii AI\04_scripts\04_p1_p2.py:59: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  g["total"]=g.sum(1); g["pct_F"]=(g.get("F",0)/g.total*100).round(1)
C:\Users\Dell\Agus\Nivii AI\04_scripts\04_p1_p2.py:65: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  n["total"]=n.sum(1); n["pct_F"]=(n.get("F",0)/n.total*100).round(1)


## 5. Verificación de posiciones críticas

Contrasta dotación reducida, roles de liderazgo y antigüedad. Así no se llama crítico a un puesto por una sola señal aislada.

**Script ejecutado:** `04_scripts/05_verif_critica.py`


In [14]:
ejecutar_etapa("05_verif_critica.py")


### es_posicion_critica
  filas panel con critica=True: 152
  empleados que ALGUNA VEZ fueron criticos: 10
  empleados criticos en su ULTIMO snapshot: 10
  varia dentro del mismo empleado?: 0 empleados

  Quienes son los criticos (activos):
 empleado_id                    area                  puesto  nivel_jerarquico  edad  antiguedad_anios  span_of_control  meses_hasta_jubilacion
        1098               Estampado           Técnico Setup                 1    64              11.3                0                       6
        1484 Mantenimiento Eléctrico          Instrumentista                 1    58               6.4                0                      66
        1542               Logística   Team Leader Logística                 2    62               6.3                1                      12
        1544               Logística   Team Leader Logística                 2    63               4.3                2                       9
        1547               Logística Su

C:\Users\Dell\Agus\Nivii AI\04_scripts\05_verif_critica.py:41: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  a['score_crit'] = a[['escaso','manda','knowhow','top']].sum(1)


## 6. carga, seguridad y rotación

Genera indicadores de horas extra, sobrecarga, incidentes y rotación. Estos indicadores conectan bienestar operativo y riesgo de salida.

**Script ejecutado:** `04_scripts/06_p3_p4_p5.py`


In [15]:
ejecutar_etapa("06_p3_p4_p5.py")


P3 - HORAS EXTRA: CONCENTRACION Y COSTO

Costo TOTAL horas extra (17 meses): $1,399,111,105 | sobre nomina base: 8.3%
Costo HE anualizado (12m): $987,607,839

--- Pareto: concentracion del costo de HE por empleado ---
  Top    5% empleados ( 33 pers.) concentran 10.7% del costo de HE
  Top   10% empleados ( 67 pers.) concentran 19.5% del costo de HE
  Top   20% empleados (135 pers.) concentran 34.6% del costo de HE
  Top   30% empleados (202 pers.) concentran 47.6% del costo de HE

--- HE por AREA (promedio mensual por empleado) ---
                         emp  he_prom_mes  he_p90  pct_he_nomina      costo_he  costo_he_anual
area                                                                                          
Mantenimiento Eléctrico   46        12.06   18.12          10.28  9.995532e+07      70556695.0
Pintura                   80        11.92   16.50          10.16  2.040502e+08     144035458.0
Logística                 35        11.50   15.20           9.80  8.690709e+07   

## 7. Verificación de incidentes y capacitación

Cruza información a nivel empleado-mes y compara fuentes. Es una auditoría para comprobar que los incidentes y la relación con capacitación son trazables.

**Script ejecutado:** `04_scripts/07_verif_incidentes_p5.py`


In [ ]:
ejecutar_etapa("07_verif_incidentes_p5.py")


## 8. Visualizaciones

Construye los gráficos de personas, equidad, carga, seguridad, rotación y trazabilidad a partir de los resultados de las etapas anteriores.

**Script ejecutado:** `04_scripts/08_visualizaciones.py`


In [ ]:
ejecutar_etapa("08_visualizaciones.py")
mostrar_graficos([
    "G1_piramide_etaria.png",
    "G2_genero_area_y_nivel.png",
    "G3_brecha_salarial.png",
    "G4_horas_extra.png",
    "G5_seguridad.png",
    "G6_rotacion.png",
    "G7_brecha_trazabilidad.png",
])


## 9. Sensibilidad de horas extra

Normaliza las ausencias por tiempo observado y prueba si la diferencia encontrada puede deberse al sesgo de exposición. Evita comparar empleados con distinta cantidad de meses registrados.

**Script ejecutado:** `04_scripts/10_sensibilidad_he.py`


In [ ]:
ejecutar_etapa("10_sensibilidad_he.py")


## 10. Verificación de sobrecarga crónica y seguridad

Contrasta la señal de sobrecarga con incidentes desde más de una fuente y aplica una prueba simple. Busca diferenciar una coincidencia casual de una relación que merece atención.

**Script ejecutado:** `04_scripts/11_verif_cronicos_seguridad.py`


In [ ]:
ejecutar_etapa("11_verif_cronicos_seguridad.py")


## 11. Diagnóstico de gaps

Identifica limitaciones del dataset y separa datos medidos de supuestos. Marca qué conclusiones son sólidas y cuáles requieren validación con el cliente.

**Script ejecutado:** `04_scripts/12_diagnostico_gaps.py`


In [ ]:
ejecutar_etapa("12_diagnostico_gaps.py")


## 12. EDA sistemático

Recorre las variables con reglas explícitas para detectar vacíos, rarezas, asimetrías y columnas poco informativas. Es una revisión ordenada para no depender solo de la intuición.

**Script ejecutado:** `04_scripts/14_eda_sistematico.py`


In [ ]:
ejecutar_etapa("14_eda_sistematico.py")


## 13. Diagnóstico de fragilidad

Examina reingresos, huecos de historial y cambios inconsistentes de antigüedad. Estos patrones pueden inflar o distorsionar indicadores de rotación.

**Script ejecutado:** `04_scripts/15_diagnostico_fragilidad.py`


In [ ]:
ejecutar_etapa("15_diagnostico_fragilidad.py")


## 14. Rotación temporal - versión mejorada

Separa las salidas de arrastre del inicio del período y recalcula la rotación sobre un período limpio. Evita atribuir al período analizado salidas que ya venían ocurriendo antes del corte.

**Script ejecutado:** `04_scripts/16_rotacion_temporal.py`


In [ ]:
ejecutar_etapa("16_rotacion_temporal.py")


## 15. Business case - versión mejorada

Presenta el impacto como un rango de escenarios y separa datos reales de supuestos. También incorpora márgenes de error para no sobreafirmar diferencias entre áreas. Esta versión reemplaza a 09_business_case.py.

**Script ejecutado:** `04_scripts/17_business_case_v2.py`


In [ ]:
ejecutar_etapa("17_business_case_v2.py")


## 16. Visualizaciones para decision

Estos siete graficos conectan la evidencia con decisiones de rotacion, seguridad, sucesion, sobrecarga y business case. Se muestran directamente debajo de esta celda y tambien se guardan como archivos PNG.

**Script ejecutado:** `04_scripts/18_visualizaciones_decision.py`


In [ ]:
ejecutar_etapa("18_visualizaciones_decision.py")
mostrar_graficos([
    "G8_rotacion_original_vs_limpia.png",
    "G9_rotacion_area_ic95.png",
    "G10_distribucion_horas_extra_area.png",
    "G11_incidentes_area_severidad.png",
    "G12_capacitacion_seguridad_vs_incidentes.png",
    "G13_riesgo_sucesion_por_puesto.png",
    "G14_business_case_escenarios.png",
])


## Cierre

Se ejecutaron las 15 etapas vigentes. Las versiones mejoradas reemplazan a las versiones anteriores de limpieza y business case.


In [ ]:
ETAPAS_EJECUTADAS = [
    "01_perfilado.py", "02_calidad.py", "13_limpieza_v2.py", "04_p1_p2.py",
    "05_verif_critica.py", "06_p3_p4_p5.py", "07_verif_incidentes_p5.py",
    "08_visualizaciones.py", "10_sensibilidad_he.py", "11_verif_cronicos_seguridad.py",
    "12_diagnostico_gaps.py", "14_eda_sistematico.py", "15_diagnostico_fragilidad.py",
    "16_rotacion_temporal.py", "17_business_case_v2.py",
    "18_visualizaciones_decision.py",
]

print(f"Etapas disponibles: {len(ETAPAS_EJECUTADAS)}")
print("Limpieza vigente: 13_limpieza_v2.py")
print("Business case vigente: 17_business_case_v2.py")
